# Dataset Adaptation Summary

Le dataset public **Brazilian E-commerce (Olist)** a été utilisé comme source de données pour ce projet, les données réelles de Carrefour ne pouvant pas être utilisées pour des raisons de confidentialité.

Afin de l’adapter à un contexte de **grande distribution et de programme de fidélité**, plusieurs transformations ont été réalisées.

Le modèle a été simplifié en transformant les **orders en transactions** et en supprimant certaines tables non pertinentes pour l’analyse (**géolocalisation**, **avis clients**, **vendeurs**).

Un **système de fidélité simulé** a été introduit avec la création :
- d’un **foyer client** (`foyer_id`)
- d’une **carte de fidélité** (`card_id`)
- d’un **statut de carte** (`card_status`)

Les **statuts de transaction** ont été regroupés en trois catégories :

- **VALIDATED**
- **CANCELLED**
- **PENDING**

Les **moyens de paiement** ont été simplifiés en :
- **card**
- **cash**
- **other**

Une variable **channel** a été ajoutée pour simuler un environnement **omnicanal** :
- `store`
- `online`

Les dates du dataset initial ont été **décalées vers une période récente (2024–2026)** et une variable **quantity** a été simulée afin de représenter un comportement d’achat réaliste en grande distribution.

À l’issue de ces transformations, trois datasets principaux ont été générés :

- **transactions_enriched**
- **transaction_items**
- **products**


In [2]:
import os
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

PROJECT_DIR = r"C:\Users\soufi\Documents\projet-carrefour"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
ARCHIVE_DIR = os.path.join(DATA_DIR, "archive")

IN_FILE = os.path.join(ARCHIVE_DIR, "olist_customers_dataset.csv")

Charger customers (Olist)

In [3]:
df = pd.read_csv(IN_FILE)
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
df.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state'], dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


Le rajout du (nom /prénom /année de naissance /email /ville /code postale /région française )

In [8]:
import numpy as np

# Pour avoir toujours le même résultat à chaque exécution
np.random.seed(42)

# --- listes de prénoms / noms français ---
prenoms = [
    "Lucas", "Emma", "Lina", "Louis", "Inès", "Adam", "Jade", "Mohamed", "Léa", "Hugo",
    "Yanis", "Sarah", "Nina", "Noah", "Sofia", "Amir", "Mila", "Chloé", "Youssef", "Aya",
    "Soumia", "Nora", "Malak", "Omar", "Hicham", "Mehdi", "Yasmine", "Nassim", "Rania", "Imane"
]

noms = [
    "Martin", "Bernard", "Dubois", "Thomas", "Robert", "Richard", "Petit", "Durand", "Leroy", "Moreau",
    "Simon", "Laurent", "Lefebvre", "Michel", "Garcia", "David", "Bertrand", "Roux", "Vincent", "Fournier",
    "Lazrak", "Benali", "ElMansouri", "AitAhmed", "Hamidi", "Bensaid", "Mouline", "Saidi", "Amrani", "Cherkaoui"
]

# --- villes françaises + code postal + région ---
villes_fr = [
    ("Paris", "75001", "Île-de-France"),
    ("Marseille", "13001", "Provence-Alpes-Côte d’Azur"),
    ("Lyon", "69001", "Auvergne-Rhône-Alpes"),
    ("Toulouse", "31000", "Occitanie"),
    ("Nice", "06000", "Provence-Alpes-Côte d’Azur"),
    ("Nantes", "44000", "Pays de la Loire"),
    ("Strasbourg", "67000", "Grand Est"),
    ("Montpellier", "34000", "Occitanie"),
    ("Bordeaux", "33000", "Nouvelle-Aquitaine"),
    ("Lille", "59000", "Hauts-de-France"),
    ("Rennes", "35000", "Bretagne"),
    ("Reims", "51100", "Grand Est"),
    ("Le Havre", "76600", "Normandie"),
    ("Saint-Étienne", "42000", "Auvergne-Rhône-Alpes"),
    ("Toulon", "83000", "Provence-Alpes-Côte d’Azur"),
    ("Grenoble", "38000", "Auvergne-Rhône-Alpes"),
    ("Dijon", "21000", "Bourgogne-Franche-Comté"),
    ("Angers", "49000", "Pays de la Loire"),
    ("Nîmes", "30000", "Occitanie"),
    ("Clermont-Ferrand", "63000", "Auvergne-Rhône-Alpes")
]

n = len(df)

# Génération prénom / nom / année de naissance
df["first_name"] = np.random.choice(prenoms, size=n)
df["last_name"] = np.random.choice(noms, size=n)
df["birth_year"] = np.random.randint(1950, 2005, size=n)

# Nettoyage simple pour email
first_clean = (
    df["first_name"]
    .str.lower()
    .str.replace(" ", "", regex=False)
)

last_clean = (
    df["last_name"]
    .str.lower()
    .str.replace(" ", "", regex=False)
)

# Génération email
df["email"] = first_clean + "." + last_clean + "+" + df["customer_id"].str[:6] + "@carrefour.fr"

# Attribution d’une ville / code postal / région française
adresses = np.random.choice(len(villes_fr), size=n)

df["customer_city"] = [villes_fr[i][0] for i in adresses]
df["postal_code"] = [villes_fr[i][1] for i in adresses]
df["region"] = [villes_fr[i][2] for i in adresses]

# Remplacement de l'ancienne logique brésilienne
df["customer_zip_code_prefix"] = df["postal_code"]
df["customer_state"] = "France"

df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,first_name,last_name,birth_year,email,postal_code,region
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,21000,Dijon,France,Jade,Bertrand,1978,jade.bertrand+06b899@carrefour.fr,21000,Bourgogne-Franche-Comté
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,51100,Reims,France,Aya,Durand,1995,aya.durand+18955e@carrefour.fr,51100,Grand Est
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,30000,Nîmes,France,Rania,Martin,1951,rania.martin+4e7b3e@carrefour.fr,30000,Occitanie
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,13001,Marseille,France,Sofia,Benali,1995,sofia.benali+b2b602@carrefour.fr,13001,Provence-Alpes-Côte d’Azur
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,83000,Toulon,France,Yanis,Lazrak,1968,yanis.lazrak+4f2d8a@carrefour.fr,83000,Provence-Alpes-Côte d’Azur


In [9]:
# création du foyer
import numpy as np

# même seed pour garder un résultat stable
np.random.seed(42)

n_clients = len(df)

# tailles de foyer possibles
tailles_possibles = [1, 2, 3, 4, 5]
probabilites = [0.30, 0.35, 0.20, 0.10, 0.05]

foyer_ids = []
compteur_clients = 0
numero_foyer = 1

while compteur_clients < n_clients:
    taille_foyer = np.random.choice(tailles_possibles, p=probabilites)
    taille_foyer = min(taille_foyer, n_clients - compteur_clients)

    foyer_code = f"FOYER_{numero_foyer:05d}"
    foyer_ids.extend([foyer_code] * taille_foyer)

    compteur_clients += taille_foyer
    numero_foyer += 1

df["foyer_id"] = foyer_ids

df[["customer_id", "first_name", "last_name", "foyer_id"]].head(10)

,customer_id,first_name,last_name,foyer_id
0,06b8999e2fba1a1fbc88172c00ba8bc7,Jade,Bertrand,FOYER_00001
1,18955e83d337fd6b2def6b18a428ac77,Aya,Durand,FOYER_00001
2,4e7b3e00288586ebd08712fdd0374a03,Rania,Martin,FOYER_00002
3,b2b6027bc5c5109e529d4dc6358b12c3,Sofia,Benali,FOYER_00002
4,4f2d8ab171c80ec8364f7c12e35b23ad,Yanis,Lazrak,FOYER_00002
5,879864dab9bc3047522c92c82e1212b8,Mohamed,Moreau,FOYER_00002
6,fd826e7cf63160e536e0908c76c3f441,Rania,Garcia,FOYER_00002
7,5e274e7a0c3809e14aba7ad5aae0d407,Soumia,ElMansouri,FOYER_00003
8,5adf08e34b2e993982a47070956c5c65,Jade,Richard,FOYER_00003
9,4b7139f34592b3a31687243a302fa75b,Mehdi,Moreau,FOYER_00003


Harmonise certaines informations pour tous les membres d’un même foyer

In [11]:
# 1) Même ville / code postal / région / état pour tous les membres du foyer
df["customer_city"] = df.groupby("foyer_id")["customer_city"].transform("first")
df["postal_code"] = df.groupby("foyer_id")["postal_code"].transform("first")
df["region"] = df.groupby("foyer_id")["region"].transform("first")
df["customer_zip_code_prefix"] = df.groupby("foyer_id")["customer_zip_code_prefix"].transform("first")
df["customer_state"] = df.groupby("foyer_id")["customer_state"].transform("first")

# 2) Même nom de famille pour tous les membres du foyer
df["last_name"] = df.groupby("foyer_id")["last_name"].transform("first")

# 3) Recalcul de l'email après harmonisation du nom
first_clean = (
    df["first_name"]
    .str.lower()
    .str.replace(" ", "", regex=False)
)

last_clean = (
    df["last_name"]
    .str.lower()
    .str.replace(" ", "", regex=False)
)

df["email"] = first_clean + "." + last_clean + "+" + df["customer_id"].str[:6] + "@carrefour.fr"

# Vérification
df[["foyer_id", "first_name", "last_name", "customer_city", "postal_code", "region", "email"]].head(15)

,foyer_id,first_name,last_name,customer_city,postal_code,region,email
0,FOYER_00001,Jade,Bertrand,Dijon,21000,Bourgogne-Franche-Comté,jade.bertrand+06b899@carrefour.fr
1,FOYER_00001,Aya,Bertrand,Dijon,21000,Bourgogne-Franche-Comté,aya.bertrand+18955e@carrefour.fr
2,FOYER_00002,Rania,Martin,Nîmes,30000,Occitanie,rania.martin+4e7b3e@carrefour.fr
3,FOYER_00002,Sofia,Martin,Nîmes,30000,Occitanie,sofia.martin+b2b602@carrefour.fr
4,FOYER_00002,Yanis,Martin,Nîmes,30000,Occitanie,yanis.martin+4f2d8a@carrefour.fr
5,FOYER_00002,Mohamed,Martin,Nîmes,30000,Occitanie,mohamed.martin+879864@carrefour.fr
6,FOYER_00002,Rania,Martin,Nîmes,30000,Occitanie,rania.martin+fd826e@carrefour.fr
7,FOYER_00003,Soumia,ElMansouri,Lyon,69001,Auvergne-Rhône-Alpes,soumia.elmansouri+5e274e@carrefour.fr
8,FOYER_00003,Jade,ElMansouri,Lyon,69001,Auvergne-Rhône-Alpes,jade.elmansouri+5adf08@carrefour.fr
9,FOYER_00003,Mehdi,ElMansouri,Lyon,69001,Auvergne-Rhône-Alpes,mehdi.elmansouri+4b7139@carrefour.fr


In [12]:
# Nombre total de foyers
df["foyer_id"].nunique()

44237

In [13]:
# Taille max d’un foyer
df.groupby("foyer_id")["customer_id"].count().max()

np.int64(5)

In [14]:
# Calcul taille de chaque foyer
foyer_sizes = df.groupby("foyer_id")["customer_id"].count()

# Comptage des tailles
distribution = foyer_sizes.value_counts().sort_index()

distribution

customer_id
1    13265
2    15568
3     8788
4     4404
5     2212
Name: count, dtype: int64

Charger orders (Olist)

In [16]:
ORDERS_FILE = os.path.join(ARCHIVE_DIR, "olist_orders_dataset.csv")
orders = pd.read_csv(ORDERS_FILE)

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [17]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

Convertir le timestamp

In [ ]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_purchase_timestamp"].min(), orders["order_purchase_timestamp"].max()

(Timestamp('2016-09-04 21:15:19'), Timestamp('2018-10-17 17:30:18'))

Créer transactions_clean 

In [19]:
transactions_clean = orders.rename(columns={
    "order_id": "transaction_id",
    "order_status": "transaction_status",
    "order_purchase_timestamp": "transaction_timestamp"
})[["transaction_id", "customer_id", "transaction_status", "transaction_timestamp"]].copy()

transactions_clean.head()

,transaction_id,customer_id,transaction_status,transaction_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39


In [20]:
transactions_clean["transaction_status"].value_counts()

transaction_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Mapper les statuts :

delivered, shipped, invoiced, approved → VALIDATED

canceled, unavailable → CANCELLED

processing, created → PENDING

In [21]:
status_map = {
    "delivered": "VALIDATED",
    "shipped": "VALIDATED",
    "invoiced": "VALIDATED",
    "approved": "VALIDATED",
    "canceled": "CANCELLED",
    "unavailable": "CANCELLED",
    "processing": "PENDING",
    "created": "PENDING"
}

transactions_clean["transaction_status"] = (
    transactions_clean["transaction_status"]
        .map(status_map)
)

In [22]:
transactions_clean["transaction_status"].value_counts()

transaction_status
VALIDATED    97901
CANCELLED     1234
PENDING        306
Name: count, dtype: int64

Joindre transactions avec customers

In [23]:
# Jointure transactions + clients

transactions_enriched = transactions_clean.merge(
    df,
    on="customer_id",
    how="left"
)

transactions_enriched.head()

,transaction_id,customer_id,transaction_status,transaction_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,first_name,last_name,birth_year,email,postal_code,region,foyer_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,VALIDATED,2017-10-02 10:56:33,7c396fd4830fd04220f754e42b4e5bff,42000,Saint-Étienne,France,Rania,David,1961,rania.david+9ef432@carrefour.fr,42000,Auvergne-Rhône-Alpes,FOYER_31265
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,VALIDATED,2018-07-24 20:41:37,af07308b275d755c9edb36a90c618231,35000,Rennes,France,Aya,ElMansouri,1966,aya.elmansouri+b0830f@carrefour.fr,35000,Bretagne,FOYER_34249
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,VALIDATED,2018-08-08 08:38:49,3a653a41f6f9fc3d2a113cf8398680e8,38000,Grenoble,France,Jade,Simon,1978,jade.simon+41ce2a@carrefour.fr,38000,Auvergne-Rhône-Alpes,FOYER_00250
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,VALIDATED,2017-11-18 19:28:06,7c142cf63193a1473d2e66489a9ae977,33000,Bordeaux,France,Omar,Leroy,1974,omar.leroy+f88197@carrefour.fr,33000,Nouvelle-Aquitaine,FOYER_27099
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,VALIDATED,2018-02-13 21:18:39,72632f0f9dd73dfee390c9b22eb56dd6,76600,Le Havre,France,Jade,Fournier,1990,jade.fournier+8ab979@carrefour.fr,76600,Normandie,FOYER_29924


In [24]:
transactions_enriched.isna().sum()

transaction_id              0
customer_id                 0
transaction_status          0
transaction_timestamp       0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
first_name                  0
last_name                   0
birth_year                  0
email                       0
postal_code                 0
region                      0
foyer_id                    0
dtype: int64

In [25]:
transactions_enriched.shape

(99441, 15)

Préparer payment_type (card/cash)

In [26]:
PAYMENTS_FILE = os.path.join(ARCHIVE_DIR, "olist_order_payments_dataset.csv")
payments = pd.read_csv(PAYMENTS_FILE)
payments.head()


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [27]:
payments["payment_type"].unique()

array(['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined'],
      dtype=object)

In [28]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Mapping pour le contexte grande distribution (Carrefour simulé) :

credit_card → carte bancaire → card

debit_card → carte bancaire → card

boleto → paiement type ticket bancaire → cash

voucher → bon d’achat → cash

not_defined → rare anomalie → other 

In [30]:
def map_payment_type(pt):
    pt = str(pt).lower().strip()

    if pt in ["credit_card", "debit_card"]:
        return "card"
    elif pt in ["boleto", "voucher"]:
        return "cash"
    else:
        return "other"

payments["payment_type_simple"] = payments["payment_type"].apply(map_payment_type)

payments["payment_type_simple"].value_counts()

payment_type_simple
card     78324
cash     25559
other        3
Name: count, dtype: int64

Agrégation du type de payement par transaction

In [37]:
payments_agg = (
    payments.groupby("order_id")["payment_type_simple"]
    .apply(lambda s: ",".join(sorted(set(s))))
    .reset_index()
    .rename(columns={
        "order_id": "transaction_id",
        "payment_type_simple": "payment_types_used"
    })
)

payments_agg.head()

,transaction_id,payment_types_used
0,00010242fe8c5a6d1ba2dd792cb16214,card
1,00018f77f2f0320c557190d7a144bdd3,card
2,000229ec398224ef6ca0657da4fc703e,card
3,00024acbcdf0a6daa1e931b038114c75,card
4,00042b26cf59d7ce69dfabb4e55b4fd9,card


In [38]:
payments_agg.shape

(99440, 2)

In [39]:
transactions_enriched = transactions_enriched.merge(
    payments_agg,
    on="transaction_id",
    how="left"
)

transactions_enriched["payment_types_used"] = (
    transactions_enriched["payment_types_used"]
    .fillna("other")
)

In [40]:
transactions_enriched.columns

Index(['transaction_id', 'customer_id', 'transaction_status', 'transaction_timestamp', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'first_name', 'last_name', 'birth_year', 'email', 'postal_code', 'region', 'foyer_id', 'payment_types_used_x', 'payment_types_used_y',
       'payment_types_used'],
      dtype='object')

In [43]:
transactions_enriched["transaction_timestamp"] = pd.to_datetime(
    transactions_enriched["transaction_timestamp"]
)

Décalage temporel du Dataset

In [44]:
import pandas as pd

max_date = transactions_enriched["transaction_timestamp"].max()
today = pd.Timestamp.today()

delta = today - max_date
delta

Timedelta('2710 days 04:53:04.703850')

In [45]:
transactions_enriched["transaction_timestamp"] = (
    transactions_enriched["transaction_timestamp"] + delta
)

In [46]:
transactions_enriched["transaction_timestamp"].min(), \
transactions_enriched["transaction_timestamp"].max()

(Timestamp('2024-02-06 02:08:23.703850'),
 Timestamp('2026-03-19 22:23:22.703850'))

*** Création logique de la date de création carte ***

La logique métier réaliste : La date de création carte = première transaction du client

In [47]:
card_creation = (
    transactions_enriched.groupby("customer_id")["transaction_timestamp"]
    .min()
    .reset_index()
    .rename(columns={"transaction_timestamp": "issued_at"})
)

In [48]:
import pandas as pd

max_date = transactions_enriched["transaction_timestamp"].max()
today = pd.Timestamp.today()

delta = today - max_date
delta

Timedelta('0 days 00:02:19.101640')

*** Générer la colonne "channel"***
Règles que je veux appliquer:

En se basant sur payment_types_used (agrégé par transaction) :

Si payment_types_used == "cash" → channel = "store"

Si payment_types_used == "card,cash" (ou cash+card) → channel = "store"

Si payment_types_used == "card" → 50% store, 50% online

(Optionnel) Si other ou valeurs rares → on met store (par défaut)

In [50]:
rng = np.random.default_rng(42)

def assign_channel(payment_types_used):
    s = str(payment_types_used).lower().strip()

    # cas cash ou mix => store
    if s == "cash" or "cash" in s:
        return "store"

    # cas card uniquement => 50/50
    if s == "card":
        return "store" if rng.random() < 0.5 else "online"

    # cas autres / inconnus
    return "store"

transactions_enriched["channel"] = transactions_enriched["payment_types_used"].apply(assign_channel)

In [52]:
transactions_enriched["channel"].value_counts(normalize=True) * 100

channel
store     61.685824
online    38.314176
Name: proportion, dtype: float64

L’identifiant de carte est généré de manière déterministe à partir du customer_id 
La logique: 
1 client = N carte valide

In [53]:
# Trier les transactions dans le temps
transactions_enriched = transactions_enriched.sort_values(
    ["customer_id", "transaction_timestamp"]
)

# Numéro d'achat par client
transactions_enriched["transaction_rank"] = (
    transactions_enriched.groupby("customer_id").cumcount() + 1
)

# Nouvelle carte tous les 3 achats (modifiable)
transactions_enriched["card_seq"] = (
    (transactions_enriched["transaction_rank"] - 1) // 3 + 1
)

# Génération card_id déterministe
transactions_enriched["card_id"] = (
    "CARD_" 
    + transactions_enriched["customer_id"].str[:8] 
    + "_" 
    + transactions_enriched["card_seq"].astype(str)
)

transactions_enriched[["customer_id", "transaction_rank", "card_seq", "card_id"]].head(15)

,customer_id,transaction_rank,card_seq,card_id
68578,00012a2ce6f8dcda20d059ce98491703,1,1,CARD_00012a2c_1
10013,000161a058600d5901f007fab4c27140,1,1,CARD_000161a0_1
65884,0001fd6190edaaf884bcaf3d49edf079,1,1,CARD_0001fd61_1
43174,0002414f95344307404f0ace7a26f1d5,1,1,CARD_0002414f_1
5888,000379cdec625522490c315e70c7a9fb,1,1,CARD_000379cd_1
73652,0004164d20a9e969af783496f3408652,1,1,CARD_0004164d_1
46156,000419c5494106c306a97b5635748086,1,1,CARD_000419c5_1
59978,00046a560d407e99b969756e0b10f282,1,1,CARD_00046a56_1
79246,00050bf6e01e69d5c0fd612f1bcfb69c,1,1,CARD_00050bf6_1
80371,000598caf2ef4117407665ac33275130,1,1,CARD_000598ca_1


Création de la colonne issued_at (pour Loyalty_cards):
date d’émission de la 1ére carte = première transaction du client

In [55]:
issued_at_map = (
    transactions_enriched
    .groupby(["customer_id", "card_id"])["transaction_timestamp"]
    .min()
    .reset_index()
    .rename(columns={"transaction_timestamp": "issued_at"})
)

issued_at_map.head()

,customer_id,card_id,issued_at
0,00012a2ce6f8dcda20d059ce98491703,CARD_00012a2c_1,2025-04-16 21:01:30.703850
1,000161a058600d5901f007fab4c27140,CARD_000161a0_1,2024-12-16 14:33:36.703850
2,0001fd6190edaaf884bcaf3d49edf079,CARD_0001fd61_1,2024-07-31 15:59:47.703850
3,0002414f95344307404f0ace7a26f1d5,CARD_0002414f_1,2025-01-16 18:02:24.703850
4,000379cdec625522490c315e70c7a9fb,CARD_000379cd_1,2025-09-02 18:35:21.703850


In [56]:
transactions_enriched = transactions_enriched.merge(
    issued_at_map,
    on=["customer_id", "card_id"],
    how="left"
)

In [58]:
transactions_enriched[["customer_id", "transaction_timestamp", "issued_at"]].head(10)

,customer_id,transaction_timestamp,issued_at
0,00012a2ce6f8dcda20d059ce98491703,2025-04-16 21:01:30.703850,2025-04-16 21:01:30.703850
1,000161a058600d5901f007fab4c27140,2024-12-16 14:33:36.703850,2024-12-16 14:33:36.703850
2,0001fd6190edaaf884bcaf3d49edf079,2024-07-31 15:59:47.703850,2024-07-31 15:59:47.703850
3,0002414f95344307404f0ace7a26f1d5,2025-01-16 18:02:24.703850,2025-01-16 18:02:24.703850
4,000379cdec625522490c315e70c7a9fb,2025-09-02 18:35:21.703850,2025-09-02 18:35:21.703850
5,0004164d20a9e969af783496f3408652,2024-09-12 13:28:16.703850,2024-09-12 13:28:16.703850
6,000419c5494106c306a97b5635748086,2025-08-02 22:40:44.703850,2025-08-02 22:40:44.703850
7,00046a560d407e99b969756e0b10f282,2025-05-20 16:01:34.703850,2025-05-20 16:01:34.703850
8,00050bf6e01e69d5c0fd612f1bcfb69c,2025-02-17 20:57:48.703850,2025-02-17 20:57:48.703850
9,000598caf2ef4117407665ac33275130,2026-01-11 17:07:39.703850,2026-01-11 17:07:39.703850


Creation de la colonne "card_status"
card_status = active ou expiré (aucune utilisation depuis 365 jours)

In [59]:
# 1. Dernière utilisation de chaque carte
last_use = (
    transactions_enriched
    .groupby(["customer_id", "card_id"])["transaction_timestamp"]
    .max()
    .reset_index()
    .rename(columns={"transaction_timestamp": "last_used_at"})
)

# 2. Ajouter dans le dataset
transactions_enriched = transactions_enriched.merge(
    last_use,
    on=["customer_id", "card_id"],
    how="left"
)

# 3. Date actuelle
today = pd.Timestamp.today()

# 4. Calcul du statut
transactions_enriched["card_status"] = (
    (today - transactions_enriched["last_used_at"]).dt.days
)

transactions_enriched["card_status"] = transactions_enriched["card_status"].apply(
    lambda x: "active" if x <= 365 else "expired"
)

# Vérification
transactions_enriched["card_status"].value_counts()

card_status
active     69523
expired    29918
Name: count, dtype: int64

Création de la colonne "foyer_created_at":
La date de création du foyer est 24h avant la première carte émise au sein du foyer.

In [60]:
# Calculer la première carte du foyer

foyer_first_card_map = (
    transactions_enriched
    .groupby("foyer_id")["issued_at"]
    .min()
)

In [61]:
# Ajouter foyer_created_at

transactions_enriched["foyer_created_at"] = (
    transactions_enriched["foyer_id"]
    .map(foyer_first_card_map)
    - pd.Timedelta(hours=24)
)

In [62]:
transactions_enriched[[
    "foyer_id",
    "issued_at",
    "foyer_created_at"
]].head(10)

,foyer_id,issued_at,foyer_created_at
0,FOYER_37710,2025-04-16 21:01:30.703850,2025-04-15 21:01:30.703850
1,FOYER_12231,2024-12-16 14:33:36.703850,2024-12-15 14:33:36.703850
2,FOYER_28340,2024-07-31 15:59:47.703850,2024-06-21 01:38:12.703850
3,FOYER_43476,2025-01-16 18:02:24.703850,2025-01-15 18:02:24.703850
4,FOYER_26577,2025-09-02 18:35:21.703850,2024-07-30 17:32:05.703850
5,FOYER_09075,2024-09-12 13:28:16.703850,2024-09-11 13:28:16.703850
6,FOYER_19834,2025-08-02 22:40:44.703850,2025-03-10 20:51:36.703850
7,FOYER_41126,2025-05-20 16:01:34.703850,2025-05-19 16:01:34.703850
8,FOYER_26068,2025-02-17 20:57:48.703850,2025-02-16 20:57:48.703850
9,FOYER_08021,2026-01-11 17:07:39.703850,2024-09-22 20:42:04.703850


Chargement du fichier "olist_order_items_dataset.csv"

In [63]:
ITEMS_FILE = os.path.join(ARCHIVE_DIR, "olist_order_items_dataset.csv")
items = pd.read_csv(ITEMS_FILE)

items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [64]:
# Renommer les colonnes
items_clean = items.rename(columns={
    "order_id": "transaction_id",
    "order_item_id": "transaction_item_id",
    "price": "unit_price"
})[["transaction_id", "transaction_item_id", "product_id", "unit_price"]].copy()

items_clean.head()

,transaction_id,transaction_item_id,product_id,unit_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,239.90
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,199.90


Logique réaliste de la quantité de produit acheté :

1 dans la majorité des cas

parfois 2–3

rarement 4–10

In [65]:
rng = np.random.default_rng(42)

def simulate_quantity(n):
    values = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
    probs  = np.array([0.85, 0.10, 0.03, 0.015, 0.004, 0.0002, 0.0002, 0.0002, 0.0002, 0.0002])
    probs = probs / probs.sum()   # sécurité : somme = 1
    return rng.choice(values, size=n, p=probs)

items_clean["quantity"] = simulate_quantity(len(items_clean))

# Vérifier la répartition
(items_clean["quantity"].value_counts(normalize=True).sort_index() * 100).round(3)

quantity
1     84.991
2      9.958
3      3.053
4      1.507
5      0.389
6      0.026
7      0.017
8      0.020
9      0.020
10     0.020
Name: proportion, dtype: float64

Lire le fichier products

In [66]:
PRODUCTS_FILE = os.path.join(ARCHIVE_DIR, "olist_products_dataset.csv")
products = pd.read_csv(PRODUCTS_FILE)

products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [67]:
products.columns
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [68]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")

In [69]:
products = products[[
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]].copy()
products.head()

,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,625.0,20.0,17.0,13.0


In [70]:
products["product_category_name"].value_counts()

product_category_name
cama_mesa_banho                  3029
esporte_lazer                    2867
moveis_decoracao                 2657
beleza_saude                     2444
utilidades_domesticas            2335
                                 ... 
fashion_roupa_infanto_juvenil       5
casa_conforto_2                     5
pc_gamer                            3
seguros_e_servicos                  2
cds_dvds_musicais                   1
Name: count, Length: 74, dtype: int64

Exporter transactions

In [71]:
transactions_enriched.columns

Index(['transaction_id', 'customer_id', 'transaction_status', 'transaction_timestamp', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'first_name', 'last_name', 'birth_year', 'email', 'postal_code', 'region', 'foyer_id', 'payment_types_used_x', 'payment_types_used_y',
       'payment_types_used', 'channel', 'transaction_rank', 'card_seq', 'card_id', 'issued_at', 'last_used_at', 'card_status', 'foyer_created_at'],
      dtype='object')

In [72]:
(transactions_enriched["payment_types_used_x"] == transactions_enriched["payment_types_used_y"]).value_counts()

True     99440
False        1
Name: count, dtype: int64

In [73]:
transactions_enriched[
    transactions_enriched["payment_types_used_x"] != transactions_enriched["payment_types_used_y"]
][[
    "payment_types_used_x",
    "payment_types_used_y",
    "transaction_id"
]]

,payment_types_used_x,payment_types_used_y,transaction_id
52277,other,NaN,bfbd0f9bdef84302105ad712db648a6c


In [ ]:
transactions_enriched = transactions_enriched.rename(columns={
    "payment_types_used_x": "payment_types_used"
})

In [75]:
transactions_enriched = transactions_enriched.drop(columns=[
    "payment_types_used_y",
    "transaction_rank",
    "card_seq",
    "customer_zip_code_prefix",
    "customer_state"
])

In [76]:
transactions_enriched.columns

Index(['transaction_id', 'customer_id', 'transaction_status', 'transaction_timestamp', 'customer_unique_id', 'customer_city', 'first_name', 'last_name',
       'birth_year', 'email', 'postal_code', 'region', 'foyer_id', 'payment_types_used', 'payment_types_used', 'channel', 'card_id', 'issued_at',
       'last_used_at', 'card_status', 'foyer_created_at'],
      dtype='object')

In [77]:
transactions_enriched = transactions_enriched.loc[:, ~transactions_enriched.columns.duplicated()]

In [84]:
# 1. supprimer l'ancien customer_id
transactions_enriched = transactions_enriched.drop(columns=["customer_id"])

# 2. renommer customer_unique_id
transactions_enriched = transactions_enriched.rename(columns={
    "customer_unique_id": "customer_id"
})

transactions_enriched.head()

,transaction_id,transaction_status,transaction_timestamp,customer_id,customer_city,first_name,last_name,birth_year,email,postal_code,region,foyer_id,payment_types_used,channel,card_id,issued_at,last_used_at,card_status,foyer_created_at
0,5f79b5b0931d63f1a42989eb65b9da6e,VALIDATED,2025-04-16 21:01:30.703850,248ffe10d632bebe4f7267f1f44844c9,Clermont-Ferrand,Mohamed,Laurent,1989,mohamed.laurent+00012a@carrefour.fr,63000,Auvergne-Rhône-Alpes,FOYER_37710,card,store,CARD_00012a2c_1,2025-04-16 21:01:30.703850,2025-04-16 21:01:30.703850,active,2025-04-15 21:01:30.703850
1,a44895d095d7e0702b6a162fa2dbeced,VALIDATED,2024-12-16 14:33:36.703850,b0015e09bb4b6e47c52844fab5fb6638,Lille,Amir,ElMansouri,1966,amir.elmansouri+000161@carrefour.fr,59000,Hauts-de-France,FOYER_12231,card,store,CARD_000161a0_1,2024-12-16 14:33:36.703850,2024-12-16 14:33:36.703850,expired,2024-12-15 14:33:36.703850
2,316a104623542e4d75189bb372bc5f8d,VALIDATED,2024-07-31 15:59:47.703850,94b11d37cd61cb2994a194d11f89682b,Toulouse,Adam,Saidi,1983,adam.saidi+0001fd@carrefour.fr,31000,Occitanie,FOYER_28340,card,store,CARD_0001fd61_1,2024-07-31 15:59:47.703850,2024-07-31 15:59:47.703850,expired,2024-06-21 01:38:12.703850
3,5825ce2e88d5346438686b0bba99e5ee,VALIDATED,2025-01-16 18:02:24.703850,4893ad4ea28b2c5b3ddf4e82e79db9e6,Strasbourg,Soumia,Simon,2001,soumia.simon+000241@carrefour.fr,67000,Grand Est,FOYER_43476,cash,store,CARD_0002414f_1,2025-01-16 18:02:24.703850,2025-01-16 18:02:24.703850,expired,2025-01-15 18:02:24.703850
4,0ab7fb08086d4af9141453c91878ed7a,VALIDATED,2025-09-02 18:35:21.703850,0b83f73b19c2019e182fd552c048a22c,Toulon,Malak,Mouline,1987,malak.mouline+000379@carrefour.fr,83000,Provence-Alpes-Côte d’Azur,FOYER_26577,cash,store,CARD_000379cd_1,2025-09-02 18:35:21.703850,2025-09-02 18:35:21.703850,active,2024-07-30 17:32:05.703850


In [87]:
transactions_enriched = transactions_enriched.rename(columns={
    "foyer_id": "household_id",
    "foyer_created_at": "household_created_at"
})

In [92]:
transactions_enriched.columns

Index(['transaction_id', 'transaction_status', 'transaction_timestamp', 'customer_id', 'customer_city', 'first_name', 'last_name', 'birth_year', 'email',
       'postal_code', 'region', 'household_id', 'payment_types_used', 'channel', 'card_id', 'issued_at', 'last_used_at', 'card_status',
       'household_created_at'],
      dtype='object')

Exporter transaction_enriched

In [89]:
transactions_enriched.to_csv(os.path.join(DATA_DIR, "transactions_enriched.csv"), index=False)

Exporter transaction_items

In [90]:
items_clean.columns

Index(['transaction_id', 'transaction_item_id', 'product_id', 'unit_price', 'quantity'], dtype='object')

In [81]:
items_clean.to_csv(os.path.join(DATA_DIR, "transaction_items_clean.csv"), index=False)

Exporter products

In [91]:
products.columns

Index(['product_id', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'], dtype='object')

In [83]:
products.to_csv(os.path.join(DATA_DIR, "products_clean.csv"), index=False)